[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/scottyUX/CSE115A-Summer/blob/main/week-2/lab-02-cursor-cli.ipynb)

# Lab 02 — Cursor CLI

## What You'll Learn

By the end of this lab you will be able to:

- Install the Cursor CLI and authenticate your account.
- Start an interactive agent session from the terminal.
- Switch between Agent, Plan, and Ask modes.
- Use shell mode to run commands inside a conversation.
- Apply key CLI parameters and slash commands.
- Run the agent non-interactively in scripts and CI pipelines.
- Push long-running tasks to Cloud Agent.

---

## Step 1 — What is the Cursor CLI?

The Cursor CLI (`agent`) gives you the full power of Cursor Agent directly from your terminal — no GUI required. You can write, review, and modify code through a conversational session, or drive the agent programmatically in scripts and CI pipelines.

### Three modes

| Mode | What it does | Access |
|---|---|---|
| **Agent** | Full tool access — reads, writes, runs commands | Default |
| **Plan** | Design-first — asks clarifying questions, builds a plan before touching code | `/plan` or `--plan` |
| **Ask** | Read-only — explores and explains without making changes | `/ask` or `--mode=ask` |

Switch between modes with `Shift+Tab` or slash commands at any point in a session.

### Two interaction styles

| Style | When to use |
|---|---|
| **Interactive session** | Normal development — describe goals, review changes, approve commands |
| **Non-interactive (`-p`)** | Scripts, CI pipelines, automation — no human in the loop |

---

## Step 2 — Installation

### macOS / Linux / WSL

Run this single command in your terminal:

In [ ]:
# Run in your terminal
# curl https://cursor.com/install -fsS | bash

### Windows (native PowerShell)

```powershell
irm 'https://cursor.com/install?win32=true' | iex
```

### Add to PATH

After installation, add `~/.local/bin` to your PATH so the `agent` command is available everywhere:

**bash:**
```bash
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc && source ~/.bashrc
```

**zsh:**
```bash
echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.zshrc && source ~/.zshrc
```

### Verify

```bash
agent --version
```

### Keep it updated

The CLI auto-updates by default. To update manually:

```bash
agent update
```

---

## Step 3 — Authentication

### Browser login (recommended)

```bash
agent login
```

This opens your browser for authentication. Credentials are stored locally. Verify your session:

```bash
agent status
```

Expected output:
```
Logged in as: you@university.edu
Plan: Pro
```

To sign out:
```bash
agent logout
```

### API key (for automation and CI)

Generate a key from the [Cursor Dashboard](https://cursor.com/dashboard) → API Keys, then:

```bash
# Set as environment variable (recommended for CI)
export CURSOR_API_KEY=your_key_here

# Or pass inline
agent --api-key your_key_here "review this file"
```

> **Headless environments:** If your environment can't open a browser, set `NO_OPEN_BROWSER=1` and manually visit the URL the CLI prints.

### Checkpoint 1
`agent status` shows your account email and Pro plan.

---

## Step 4 — Your First Session

Start an interactive agent session from inside your project folder:

```bash
cd CSE115A-Summer
agent
```

You'll see the agent prompt. Try your first interaction:

```
What files are in this project and what does each one do?
```

### Essential keyboard shortcuts

| Shortcut | Action |
|---|---|
| `Shift+Tab` | Rotate between Agent / Plan / Ask modes |
| `Ctrl+R` | Review proposed file changes |
| `Arrow Up` | Cycle through previous messages |
| `Shift+Enter` | Add a newline without submitting (terminal-dependent) |
| `Ctrl+D` (x2) | Exit the CLI |

### @ mentions work in the CLI too

```
@week-1/README.md summarize the lab goals in three bullet points
```

### Checkpoint 2
The agent listed and described the project files.

---

## Step 5 — Switching Modes

Inside a session, switch modes without restarting.

### Ask mode — explore without changes

```
/ask
```
```
Explain how calculate_grade works in week-1/grade_calculator.py
```

The agent reads and explains — it will not modify any files.

### Plan mode — design before building

```
/plan
```
```
I want to add a letter_grade_distribution function to grade_calculator.py
that takes a list of scores and returns a dict counting how many A, B, C, D, F grades there are.
```

Plan mode asks clarifying questions and generates a TODO list. Nothing is written until you approve and switch back to Agent mode.

### Switch back to Agent mode

```
Shift+Tab
```

Or type your implementation prompt directly — the mode indicator in the prompt shows which mode you're in.

### Checkpoint 3
You used Ask mode to explore a file and Plan mode to generate a plan — no files were modified in either.

---

## Step 6 — Shell Mode

Shell mode lets you run terminal commands from inside your agent conversation — without leaving the session.

### Enter shell mode

```
/shell
```
or use the alias `/sh`.

### Run commands

```bash
ls week-1/
```
```bash
python -m pytest week-1/ -v
```

### Important constraints

| Constraint | Detail |
|---|---|
| **Timeout** | Commands time out after 30 seconds — cannot be changed |
| **No persistent directory changes** | Each command runs independently — use `cd subdir && command` to chain |
| **No long-running processes** | Servers, interactive prompts, and input-dependent commands are not supported |
| **Output truncation** | Large outputs are automatically truncated |

**Best for:** status checks, quick builds, file operations, running tests, environment inspection.

### Exit shell mode

Press `Escape` with empty input, or `Backspace` on a blank line.

### Exercise

Enter shell mode and run the following — chain them in one command:

```bash
cd week-1 && ls -la
```

Then exit shell mode and ask the agent:

```
Based on the files you just saw, what's missing from week-1/ that should be there?
```

### Checkpoint 4
You ran a shell command inside an agent session and used the output as context for a follow-up prompt.

---

## Step 7 — Key Parameters

Parameters let you control how the agent starts and behaves — useful for scripting, CI, and automation.

### Most useful flags

| Flag | What it does | Example |
|---|---|---|
| `-p, --print` | Non-interactive mode — print response and exit | `agent -p "explain this file"` |
| `--mode` | Start in a specific mode | `agent --mode=ask` |
| `--model` | Select a specific model | `agent --model claude-sonnet-4-6` |
| `--resume` | Resume a previous conversation | `agent --resume` |
| `-w, --worktree` | Run in an isolated git worktree | `agent -w feature-branch` |
| `--workspace` | Set working directory | `agent --workspace /path/to/project` |
| `--output-format` | Output as `text`, `json`, or `stream-json` | `agent -p "list files" --output-format json` |
| `--api-key` | Authenticate with an API key | `agent --api-key $KEY "task"` |

### Non-interactive mode (`-p`)

Use `-p` to run the agent as part of a script or CI pipeline:

```bash
agent -p "review week-1/grade_calculator.py for missing type annotations"
```

With JSON output for parsing:

```bash
agent -p "list all functions in week-1/grade_calculator.py" --output-format json
```

### Worktrees — isolated changes

The `-w` flag creates an isolated git worktree so the agent's changes don't affect your current branch:

```bash
agent -w experiment "refactor grade_calculator.py to use a dataclass"
```

Review the changes on the `experiment` branch before merging.

### Exercise

Run the agent non-interactively to do a quick review:

```bash
agent -p "@week-1/grade_calculator.py check for missing docstrings and type annotations. List each issue on its own line."
```

### Checkpoint 5
The agent returned a non-interactive review of `grade_calculator.py` directly in the terminal.

---

## Step 8 — Essential Slash Commands

Slash commands control the session from inside the chat. Here are the most important ones:

### Session management

| Command | What it does |
|---|---|
| `/clear` | Start a fresh session (aliases: `/new`, `/new-chat`) |
| `/resume` | Open recent sessions and resume one |
| `/fork` | Branch the current conversation |
| `/rewind` | Go back to an earlier message |
| `/rename <name>` | Name the current session |

### Mode and model

| Command | What it does |
|---|---|
| `/plan [prompt]` | Switch to Plan mode |
| `/ask` | Toggle Ask mode |
| `/debug [prompt]` | Toggle Debug mode |
| `/model` | Select a different model |
| `/max-mode` | Enable Max Mode |

### Context and tools

| Command | What it does |
|---|---|
| `/summarize` | Compress conversation to free up context window (alias: `/compress`) |
| `/context` | Show how much context each resource is consuming |
| `/shell [cmd]` | Enter shell mode (aliases: `/sh`, `/run`) |
| `/mcp list` | List connected MCP servers and available tools |

### Utilities

| Command | What it does |
|---|---|
| `/open` | Open the repo root in the Cursor GUI (alias: `/cursor`) |
| `/about` | Show version, system info, and account details |
| `/config` | Adjust CLI settings interactively |
| `/help [command]` | General help or help for a specific command |
| `/feedback <msg>` | Send feedback to the Cursor team |
| `/quit` | Exit the CLI |

### Exercise — Session hygiene

1. Run `/context` — see what's consuming your context window.
2. Run `/summarize` — compress the conversation.
3. Run `/context` again — confirm the context shrank.
4. Run `/rename cli-lab` — name your session.
5. Run `/clear` — start fresh, then run `/resume` to return to the named session.

### Checkpoint 6
You managed context, named a session, cleared it, and resumed it.

---

## Step 9 — Cloud Agent Handoff

For long-running tasks, push the conversation to a **Cloud Agent** so it keeps running while you're away — no need to keep your terminal open.

### Hand off to cloud

Prepend `&` to any message:

```
& run the full test suite, fix any failing tests, and commit the result
```

The CLI returns immediately. The cloud agent continues working in the background. Check results via the Cursor web or mobile app.

### Or use `/in-cloud`

```
/in-cloud refactor all functions in week-1/ to use dataclasses
```

### Resume a cloud session

```bash
agent resume
```

> **When to use:** Any task that takes more than a few minutes — large refactors, full test suite runs, multi-file changes. Don't tie up your local terminal.

---

## Lab Completion Checklist

| # | Task | Done |
|---|---|---|
| 1 | Cursor CLI installed and `agent --version` returns a version | |
| 2 | `agent status` shows Pro plan and your `.edu` email | |
| 3 | Interactive session started — agent described the project files | |
| 4 | Switched between Agent, Plan, and Ask modes | |
| 5 | Used shell mode to run a command and used the output as context | |
| 6 | Ran a non-interactive review with `-p` | |
| 7 | Used `/summarize`, `/rename`, `/clear`, and `/resume` | |

---

## Reflection Questions

1. When would you choose the CLI over the Cursor GUI, and vice versa?
2. How would you use non-interactive mode (`-p`) in a CI pipeline for this course?
3. What is the difference between `/fork` and `/resume` — when would you use each?

---

## Resources

- [Cursor CLI Docs](https://cursor.com/docs/cli/overview)
- [Slash Commands Reference](https://cursor.com/docs/cli/reference/slash-commands)
- [CLI Parameters Reference](https://cursor.com/docs/cli/reference/parameters)
- [Week 2 README](./README.md)
- [Lab 01 — Cursor GUI](../week-1/lab-01-intro-to-cursor.ipynb)